# 08 - Composição Exploratória de Tripletos e Coocorrência de Motivos

Analisa tripletos de fragmentos sem ORFs verificadas, sinais ATG/STOP em motivos de 7 bases,
coocorrência de Jaccard e concordância entre variantes. Consome artefatos do notebook 07.
Os escores de tripletos são composicionais; nenhum resultado prova tradução, função ou causalidade.
As comparações entre matrizes são condicionadas aos motivos compartilhados e ao mesmo conjunto de sequências.

In [1]:
import os
import sys
import json
from pathlib import Path

current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from viral_cnn.m05_workflow import (
    begin_m05,
    sample_deterministic_test_sequences,
    generate_network_edge_table,
    render_figure_04_rscu,
    render_figure_05_network,
    render_figure_05_cooccurrence_synergy,
    render_figure_05_permutation_null,
    export_reconciled_table_11b_diagnostics,
    PROTOCOL_ID,
    BIOLOGICAL_DEMARCATION_TEXT,
)
from viral_cnn.interpretability.motifs import (
    compute_exploratory_triplet_rscu,
    audit_translation_boundary_signals,
    analyze_cooccurrence_pairs,
    compute_jaccard_cooccurrence,
)

m05_run = begin_m05()
ctx = m05_run.ctx
output_dir = m05_run.output_dir
motifs_07_dir = m05_run.motifs_07_dir

print(f"M05 Run ID: {m05_run.run_id}")
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir.resolve()}")
print(f"07 Insumos Dir: {motifs_07_dir.resolve() if motifs_07_dir else 'N/A'}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs


## 2. Composição Exploratória de Tripletos e RSCU (Tabela 9 e Figura 4)

Avaliação da frequência dos 64 tripletos sob três regimes de leitura: `frame0` (reprodução histórica),
`forward_3frames` e `six_frames`, com advertência científica formal de composição exploratória.


In [2]:
with m05_run.guard():
    test_parquet = ctx.topology.paths["short"].get("species_parquet_dir")
    dna_sequences = sample_deterministic_test_sequences(
        test_parquet,
        max_rows=1000,
        seed=42,
        is_smoke_test=ctx.is_smoke_test,
    )

    rscu_frame0 = compute_exploratory_triplet_rscu(dna_sequences, frame_mode="frame0")
    rscu_3f = compute_exploratory_triplet_rscu(dna_sequences, frame_mode="forward_3frames")
    rscu_6f = compute_exploratory_triplet_rscu(dna_sequences, frame_mode="six_frames")

    t9_records = []
    for codon, val_f0 in rscu_frame0["rscu"].items():
        t9_records.append({
            "codon": codon,
            "rscu_frame0": val_f0,
            "rscu_forward_3frames": rscu_3f["rscu"][codon],
            "rscu_six_frames": rscu_6f["rscu"][codon],
            "multinomial_null": rscu_frame0["multinomial_null_probabilities"][codon],
        })

    table_09_df = pd.DataFrame(t9_records)
    table_09_path = output_dir / "table_09_exploratory_rscu.csv"
    table_09_df.to_csv(table_09_path, index=False)
    print("--- TABELA 9: Composição Exploratória de Tripletos ---")
    print(table_09_df.head(5).to_string(index=False))

    fig4_path = output_dir / "figure_04_rscu_profile.png"
    render_figure_04_rscu(rscu_frame0, rscu_3f, rscu_6f, fig4_path)
    print(f"Figura 4 salva em: {fig4_path}")


[LOG_PATH] Loading test sequences from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet


[LOG_PATH] Loading ParquetSample from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/test/part.0.parquet ... (256 files total) | split=test, max_rows=1000
/tmp/ipykernel_1448023/564385034.py:28: UserWarning: Computing triplet frequencies on metagenomic fragments without verified open reading frames; treating strictly as ExploratoryTripletComposition and not proven biological codon usage.
  rscu_frame0 = compute_exploratory_triplet_rscu(dna_sequences, frame_mode="frame0")
/tmp/ipykernel_1448023/564385034.py:29: UserWarning: Computing triplet frequencies on metagenomic fragments without verified open reading frames; treating strictly as ExploratoryTripletComposition and not proven biological codon usage.
  rscu_3f = compute_exploratory_triplet_rscu(dna_sequences, frame_mode="forward_3frames")
/tmp/ipykernel_1448023/564385034.py:30: UserWarning: Computing triplet frequencies on metagenomic fragments without verified open reading frames; treating strictly as Exp

--- TABELA 9: Composição Exploratória de Tripletos ---
codon  rscu_frame0  rscu_forward_3frames  rscu_six_frames  multinomial_null
  GCA     1.361337              1.392665         1.371910          0.013060
  GCC     0.894309              0.883894         0.902760          0.009543
  GCG     0.533875              0.523958         0.521851          0.009666
  GCT     1.210479              1.199484         1.203478          0.012914
  AGA     2.101355              2.067080         2.068768          0.017874
Figura 4 salva em: results_reproduction/motifs/figure_04_rscu_profile.png


## 3. Ocorrência de ATG e STOP em motivos (Tabela 10)

Conta tripletos em motivos de 7 bases contra um modelo simples de composição de bases.
Essas ocorrências não identificam início ou término de tradução sem ORFs validadas.

In [3]:
with m05_run.guard():
    motif_summary_path = (motifs_07_dir / "motif_summary.parquet") if motifs_07_dir else (output_dir / "motif_summary.parquet")
    if not ctx.is_smoke_test and not motif_summary_path.exists():
        raise FileNotFoundError(f"Required motif_summary artifact not found at: {motif_summary_path}")

    if motif_summary_path.exists():
        df_motifs = pd.read_parquet(motif_summary_path)
        sample_motifs = sorted(list(df_motifs["motif"].unique()))
    else:
        sample_motifs = ["ATGAAAA", "TAACCCC", "TAGGGGG", "TGATTTT", "CCCCCCC", "GGGGGGG", "ATATATA"]

    base_freqs = rscu_frame0["base_frequencies"]
    boundary_audit = audit_translation_boundary_signals(sample_motifs, base_freqs)

    t10_records = []
    for key, data in boundary_audit.items():
        t10_records.append({
            "signal": key,
            "codon": data["codon"],
            "observed_count": data["observed_count"],
            "expected_count": round(data["expected_count"], 2),
            "fold_enrichment": round(data["fold_enrichment"], 2),
            "null_probability": round(data["null_probability"], 4),
        })

    table_10_df = pd.DataFrame(t10_records)
    table_10_path = output_dir / "table_10_translation_boundary_signals.csv"
    table_10_df.to_csv(table_10_path, index=False)
    print("")
    print("--- TABELA 10: Ocorrências de ATG e STOP em motivos sem ORF confirmada ---")
    print(table_10_df.to_string(index=False))



--- TABELA 10: Ocorrências de ATG e STOP em motivos sem ORF confirmada ---
   signal codon  observed_count  expected_count  fold_enrichment  null_probability
start_atg   ATG             1.0            3.45             0.29            0.0177
 stop_taa   TAA             1.0            4.66             0.21            0.0239
 stop_tag   TAG             3.0            3.45             0.87            0.0177
 stop_tga   TGA             1.0            3.45             0.29            0.0177


## 4. Coocorrência de Jaccard e escores composicionais (Tabela 11 e Figura 5)

Cada motivo de 7 bases recebe a **média dos escores exploratórios de seus cinco tripletos sobrepostos**.
Esta é uma estatística composicional sem quadro de tradução conhecido. Selecionam-se até 200
pares de motivos com Jaccard positivo; permutam-se os escores dos nós, preservando as arestas.
A estatística e seu modelo nulo usam as duas orientações de cada aresta.

In [4]:
with m05_run.guard():
    jaccard_path = (motifs_07_dir / "motif_interaction_jaccard_resnet50_species.parquet") if motifs_07_dir else (output_dir / "motif_interaction_jaccard_resnet50_species.parquet")
    if not ctx.is_smoke_test and not jaccard_path.exists():
        raise FileNotFoundError(f"Required Jaccard matrix not found at: {jaccard_path}")

    if not jaccard_path.exists():
        synthetic_motifs = ["CCCCCCC", "GGGGGGG", "ATATATA", "TTTTTTT", "AAAAAAA"]
        presence = np.random.RandomState(42).binomial(1, 0.4, size=(40, len(synthetic_motifs)))
        jacc_df = compute_jaccard_cooccurrence(presence, synthetic_motifs)
    else:
        print(f"[LOG_PATH] Loading Jaccard matrix from: {jaccard_path.resolve()}")
        jacc_df = pd.read_parquet(jaccard_path)

    print(f"Matriz de Jaccard carregada com {jacc_df.shape[0]} motivos.")
    rscu_map = rscu_frame0["rscu"]
    motif_scores = {
        motif: float(np.mean([rscu_map[motif[k:k+3]] for k in range(len(motif) - 2)]))
        for motif in jacc_df.index
        if len(motif) == 7 and set(motif).issubset(set("ACGT"))
    }
    cooccurrence_results = analyze_cooccurrence_pairs(
        jaccard_df=jacc_df,
        rscu_map=motif_scores,
        top_k=200,
        n_permutations=1000,
        seed=42,
    )

    table_11_df = pd.DataFrame([{
        "status": cooccurrence_results["status"],
        "motif_score": "mean_exploratory_triplet_rscu_in_7mer",
        "n_valid_edges": cooccurrence_results["n_valid_edges"],
        "n_valid_permutations": cooccurrence_results.get("n_valid_permutations", 0),
        "r_canonical": cooccurrence_results["r_canonical"],
        "r_reversed": cooccurrence_results["r_reversed"],
        "delta_r_orientation": cooccurrence_results["delta_r_orientation"],
        "icc_symmetric": cooccurrence_results["icc_symmetric"],
        "r_bidirectional": cooccurrence_results["r_bidirectional"],
        "p_permutation_null": cooccurrence_results["p_perm"],
    }])
    table_11_path = output_dir / "table_11_cooccurrence_synergy.csv"
    table_11_df.to_csv(table_11_path, index=False)
    print("--- TABELA 11: Coocorrência e similaridade composicional exploratória ---")
    print(table_11_df.to_string(index=False))

    edge_table_df = generate_network_edge_table(jacc_df, top_k=200, seed=42)
    edge_table_path = output_dir / "edge_table.csv"
    edge_table_df.to_csv(edge_table_path, index=False)
    print(f"Tabela de arestas C06 salva em: {edge_table_path}")

    fig5_synergy_path = output_dir / "figure_05_cooccurrence_synergy.png"
    render_figure_05_cooccurrence_synergy(cooccurrence_results, fig5_synergy_path)

    fig5_network_path = output_dir / "figure_05_network.png"
    render_figure_05_network(edge_table_df, fig5_network_path)

    fig5_null_path = output_dir / "figure_05_permutation_null.png"
    render_figure_05_permutation_null(cooccurrence_results, fig5_null_path)


[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_resnet50_species.parquet
Matriz de Jaccard carregada com 15 motivos.
--- TABELA 11: Coocorrência e similaridade composicional exploratória ---
status                           motif_score  n_valid_edges  n_valid_permutations  r_canonical  r_reversed  delta_r_orientation  icc_symmetric  r_bidirectional  p_permutation_null
 VALID mean_exploratory_triplet_rscu_in_7mer             37                  1000      0.13481     0.13481                  0.0       0.133853         0.120375             0.46953
Figura 5 salva em: results_reproduction/motifs/figure_05_cooccurrence_synergy.png


## 5. Sobreposição de motivos e diagnóstico de coocorrência entre variantes

O notebook 07 calcula presenças dos 7-mers nas sequências do mesmo `eval_loader`,
sem usar ativações do modelo na contagem. Para um motivo compartilhado, matrizes de
coocorrência obtidas sobre as mesmas sequências devem coincidir. Portanto, uma
correlação perfeita nessa situação **não testa concordância independente de modelos**.

Esta seção descreve a sobreposição dos conjuntos de motivos selecionados (até 15 por
variante) e compara os pares de Jaccard fora da diagonal. Não calcula valor de p nem FDR.
Confira o manifesto do notebook 07 para a proveniência dos artefatos comparados.

In [5]:
with m05_run.guard():
    var_keys = [
        "resnet50_species", "resnet_noise_species", "convnext_zero_species",
        "convnext_transfer_species", "convnext_noise_species",
    ]
    matrices = {}
    for vk in var_keys:
        p = (motifs_07_dir / f"motif_interaction_jaccard_{vk}.parquet") if motifs_07_dir else (output_dir / f"motif_interaction_jaccard_{vk}.parquet")
        if p.exists():
            matrices[vk] = pd.read_parquet(p)
        elif not ctx.is_smoke_test:
            raise FileNotFoundError(f"Missing Jaccard matrix for {vk} at {p}")

    variant_diagnostics_path = output_dir / "table_11b_jaccard_variant_diagnostics.csv"
    df_corrs = export_reconciled_table_11b_diagnostics(matrices, variant_diagnostics_path)
    print("--- Sobreposição de motivos e diagnóstico de Jaccard entre variantes ---")
    print(df_corrs.to_string(index=False))
    print(f"Tabela descritiva salva em: {variant_diagnostics_path}")


[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_resnet50_species.parquet
[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_resnet_noise_species.parquet
[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_convnext_zero_species.parquet
[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_convnext_transfer_species.parquet
[LOG_PATH] Loading Jaccard matrix from: /home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short/results_reproduction/motifs/motif_interaction_jaccard_convnext_noise_species.parquet
--- Sobreposição de motivos e diagnóstico de Jaccar

In [6]:
import numpy as np
import pandas as pd

pairs = [
    ("resnet50_species", "resnet_noise_species"),
    ("convnext_transfer_species", "convnext_noise_species"),
]

for name_a, name_b in pairs:
    a = pd.read_parquet(output_dir / f"motif_interaction_jaccard_{name_a}.parquet")
    b = pd.read_parquet(output_dir / f"motif_interaction_jaccard_{name_b}.parquet")
    common = sorted(set(a.index) & set(b.index))
    i, j = np.triu_indices(len(common), k=1)
    x = a.loc[common, common].to_numpy()[i, j]
    y = b.loc[common, common].to_numpy()[i, j]

    print(name_a, "×", name_b)
    print("Motivos comuns:", len(common), "| pares únicos:", len(x))
    print("Pares com Jaccard > 0:", np.count_nonzero(x), np.count_nonzero(y))
    print("Desvios padrão:", np.std(x), np.std(y))
    print("Valores iguais fora da diagonal:", np.allclose(x, y))

resnet50_species × resnet_noise_species
Motivos comuns: 3 | pares únicos: 3
Pares com Jaccard > 0: 0 0
Desvios padrão: 0.0 0.0
Valores iguais fora da diagonal: True
convnext_transfer_species × convnext_noise_species
Motivos comuns: 9 | pares únicos: 36
Pares com Jaccard > 0: 13 13
Desvios padrão: 0.01861422043586344 0.01861422043586344
Valores iguais fora da diagonal: True


## 6. Emissão do Manifesto de Execução Rastreável


In [7]:
input_records = []
if motifs_07_dir and motifs_07_dir.exists():
    for f in sorted(motifs_07_dir.glob("*.parquet")) + list(motifs_07_dir.glob("*.json")):
        if f.is_file():
            input_records.append(f)

manifest = m05_run.complete(input_files=input_records)
print(f"Manifesto 08 publicado com sucesso com status: {manifest['status']}")


Manifesto 08 salvo em: results_reproduction/motifs/manifest_08_triplets.json
